In [1]:
from pathlib import Path
import os

print("=== Searching for Sartaj Dataset ===")
for root, dirs, files in os.walk("/kaggle/input"):
    if 'sartaj' in root.lower() or 'brain-tumor-classification' in root.lower():
        print(f"Found potential path: {root}")
        path = Path(root)
        print("Contents:")
        for item in list(path.iterdir())[:10]:
            print(f"  📁 {item.name} ({'dir' if item.is_dir() else 'file'})")
        print("-" * 40)

=== Searching for Sartaj Dataset ===
Found potential path: /kaggle/input/datasets/sartajbhuvaji
Contents:
  📁 brain-tumor-classification-mri (dir)
----------------------------------------
Found potential path: /kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri
Contents:
  📁 Testing (dir)
  📁 Training (dir)
----------------------------------------
Found potential path: /kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Testing
Contents:
  📁 glioma_tumor (dir)
  📁 meningioma_tumor (dir)
  📁 no_tumor (dir)
  📁 pituitary_tumor (dir)
----------------------------------------
Found potential path: /kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Testing/glioma_tumor
Contents:
  📁 image(1).jpg (file)
  📁 image(10).jpg (file)
  📁 image(100).jpg (file)
  📁 image(11).jpg (file)
  📁 image(12).jpg (file)
  📁 image(13).jpg (file)
  📁 image(14).jpg (file)
  📁 image(15).jpg (file)
  📁 image(16).jpg (file)
  📁 image(17).jpg (file)
-------------------

In [4]:
   from kaggle_secrets import UserSecretsClient
   user_secrets = UserSecretsClient()
   hf_token = user_secrets.get_secret("HF_TOKEN")
   print("✅ Secret connected successfully!")

✅ Secret connected successfully!


In [8]:
import os
import pandas as pd
import numpy as np
import json
import hashlib
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ==========================================
# 1. CONFIGURATION
# ==========================================
SARTAJ_PATH = Path("/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri") 
HF_REPO_ID = "srijan2715/sartaj-paper-pipeline" 
SEED = 42

label_to_int = {"glioma": 0, "meningioma": 1, "no": 2, "pituitary": 3}

# ==========================================
# 2. SCAN, HASH, AND CLEAN (The Missing Step!)
# ==========================================
print("Scanning folders and calculating hashes...")
records = []

for split_name in ['Training', 'Testing']:
    split_path = SARTAJ_PATH / split_name
    if not split_path.exists(): continue
        
    for class_dir in split_path.iterdir():
        if not class_dir.is_dir(): continue
        class_name = class_dir.name.lower().replace("_tumor", "").replace("notumor", "no")
        
        for img_path in class_dir.glob("*.*"):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                with open(img_path, 'rb') as f:
                    file_hash = hashlib.sha256(f.read()).hexdigest()
                records.append({
                    "split": "train" if split_name == "Training" else "test",
                    "image_path": str(img_path),
                    "label_str": class_name,
                    "hash": file_hash
                })

df = pd.DataFrame(records)
print(f"Total raw images: {len(df)}")

# CRITICAL FIX: Remove Data Leakage & Duplicates
train_hashes = set(df[df['split']=='train']['hash'])
df_clean = df[~((df['split'] == 'test') & df['hash'].isin(train_hashes))] # Remove test images in train
df_clean = df_clean.drop_duplicates(subset=['hash']) # Remove internal duplicates
print(f"✅ Clean dataset size: {len(df_clean)} (Removed {len(df) - len(df_clean)} leaks/duplicates)")

# ==========================================
# 3. SPLIT DATA
# ==========================================
train_df = df_clean[df_clean['split'] == 'train'].copy()
test_df = df_clean[df_clean['split'] == 'test'].copy()

train_split, val_split = train_test_split(
    train_df, test_size=0.15, stratify=train_df['label_str'], random_state=SEED
)
print(f"Final Splits: Train={len(train_split)}, Val={len(val_split)}, Test={len(test_df)}")

# ==========================================
# 4. PREPROCESS (PAPER EXACT MATCH)
# ==========================================
def process_images(df, split_name):
    images, labels_int = [], []
    failed_count = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        if row['label_str'] not in label_to_int:
            failed_count += 1
            continue
            
        try:
            img = Image.open(row['image_path']).convert("RGB")
            # PAPER EXACT MATCH: Direct resize to 224x224
            img_final = img.resize((224, 224), Image.Resampling.BICUBIC)
            
            images.append(np.array(img_final))
            labels_int.append(label_to_int[row['label_str']])
        except Exception as e:
            failed_count += 1
            
    print(f"Failed/skipped in {split_name}: {failed_count}")
    return images, labels_int

print("\nResizing to 224x224...")
train_imgs, train_lbls = process_images(train_split, "train")
val_imgs, val_lbls = process_images(val_split, "val")
test_imgs, test_lbls = process_images(test_df, "test")

# ==========================================
# 5. UPLOAD
# ==========================================
with open("label_mapping.json", "w") as f:
    json.dump(label_to_int, f, indent=2)

dataset_dict = DatasetDict({
    "train": Dataset.from_dict({"image": train_imgs, "label": train_lbls}),
    "val": Dataset.from_dict({"image": val_imgs, "label": val_lbls}),
    "test": Dataset.from_dict({"image": test_imgs, "label": test_lbls})
})

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

dataset_dict.push_to_hub(HF_REPO_ID, token=hf_token, private=False, max_shard_size="500MB")
print(f"\n🎉 DONE! Link: https://huggingface.co/datasets/{HF_REPO_ID}")

Scanning folders and calculating hashes...
Total raw images: 3264
✅ Clean dataset size: 3027 (Removed 237 leaks/duplicates)
Final Splits: Train=2360, Val=417, Test=250

Resizing to 224x224...


Processing train: 100%|██████████| 2360/2360 [00:17<00:00, 133.29it/s]


Failed/skipped in train: 0


Processing val: 100%|██████████| 417/417 [00:03<00:00, 130.93it/s]


Failed/skipped in val: 0


...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.



🎉 DONE! Link: https://huggingface.co/datasets/srijan2715/sartaj-paper-pipeline


In [1]:
import hashlib
from pathlib import Path

SARTAJ_PATH = Path("/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri")

def get_file_hash(img_path):
    with open(img_path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

print("Calculating hashes for Training set...")
train_hashes = {}
train_duplicates = 0
for img_path in SARTAJ_PATH.glob("Training/*/*.jpg"):
    h = get_file_hash(img_path)
    if h in train_hashes:
        train_duplicates += 1
    else:
        train_hashes[h] = img_path

print(f"Training set scanned. Found {train_duplicates} internal duplicates.")

print("\nCalculating hashes for Testing set...")
test_hashes = {}
test_duplicates = 0
for img_path in SARTAJ_PATH.glob("Testing/*/*.jpg"):
    h = get_file_hash(img_path)
    if h in test_hashes:
        test_duplicates += 1
    else:
        test_hashes[h] = img_path

print(f"Testing set scanned. Found {test_duplicates} internal duplicates.")

# THE CRITICAL CHECK: Train/Test Overlap
print("\n=== CHECKING FOR DATA LEAKAGE (Train/Test Overlap) ===")
overlap = set(train_hashes.keys()) & set(test_hashes.keys())
print(f"Images found in BOTH Train and Test: {len(overlap)}")

if len(overlap) > 0:
    print("WARNING: Data leakage detected! These test images are already in the training set:")
    for h in list(overlap)[:5]: # Show first 5
        print(f"  Test: {test_hashes[h].name} | Train: {train_hashes[h].name}")
else:
    print("✅ No overlap found. Dataset is clean!")

Calculating hashes for Training set...
Training set scanned. Found 93 internal duplicates.

Calculating hashes for Testing set...
Testing set scanned. Found 80 internal duplicates.

=== CHECKING FOR DATA LEAKAGE (Train/Test Overlap) ===
Images found in BOTH Train and Test: 64
  Test: image(39).jpg | Train: image(14).jpg
  Test: image(30).jpg | Train: image(6).jpg
  Test: image(97).jpg | Train: image(262).jpg
  Test: image(65).jpg | Train: image(40).jpg
  Test: image(25).jpg | Train: image(1).jpg


In [3]:
print("Unique labels found in dataset:")
print(df_clean['label_str'].unique())

Unique labels found in dataset:
['no' 'pituitary' 'meningioma' 'glioma']


In [5]:
from datasets import load_dataset

print("Loading Sartaj dataset from Hugging Face...")
ds = load_dataset("srijan2715/sartaj-paper-pipeline")

print("\n=== Verifying Sartaj Image Shapes ===")
for split in ds.keys():
    print(f"Checking {split} split...")
    
    shapes = set()
    # Check the first 10 images
    for i in range(min(10, len(ds[split]))):
        img_data = ds[split][i]['image']
        img_array = np.array(img_data)
        shapes.add(img_array.shape)
        
    print(f"  Unique shapes found: {shapes}")

print("\n=== Sartaj Dataset Stats ===")
print(f"Train samples: {len(ds['train'])}")
print(f"Val samples: {len(ds['val'])}")
print(f"Test samples: {len(ds['test'])}")

print("\n✅ SARTAJ VERIFICATION COMPLETE!")

Loading Sartaj dataset from Hugging Face...


README.md:   0%|          | 0.00/536 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2360 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/417 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]


=== Verifying Sartaj Image Shapes ===
Checking train split...
  Unique shapes found: {(224, 224, 3)}
Checking val split...
  Unique shapes found: {(224, 224, 3)}
Checking test split...
  Unique shapes found: {(224, 224, 3)}

=== Sartaj Dataset Stats ===
Train samples: 2360
Val samples: 417
Test samples: 250

✅ SARTAJ VERIFICATION COMPLETE!


In [9]:
import numpy as np
import hashlib
from datasets import load_dataset

print("📦 Loading Sartaj dataset DIRECTLY from Hugging Face servers...")
# This forces a download from HF, ignoring any local cache
ds = load_dataset("srijan2715/sartaj-paper-pipeline", download_mode="force_redownload")

def get_image_hash(img_array):
    # Hash the raw bytes of the numpy array to detect exact pixel duplicates
    return hashlib.sha256(img_array.tobytes()).hexdigest()

print("\n" + "="*70)
print("SARTAJ DATASET DUPLICATE & LEAKAGE CHECK (VERIFIED FROM HF)")
print("="*70)

train_hashes = set()
val_hashes = set()
test_hashes = set()

train_duplicates = 0
val_duplicates = 0
test_duplicates = 0

# 1. Check Train Split
print("🔍 Checking Train split...")
for i in range(len(ds['train'])):
    img_array = np.array(ds['train'][i]['image'])
    h = get_image_hash(img_array)
    if h in train_hashes:
        train_duplicates += 1
    else:
        train_hashes.add(h)

# 2. Check Val Split
print("🔍 Checking Val split...")
for i in range(len(ds['val'])):
    img_array = np.array(ds['val'][i]['image'])
    h = get_image_hash(img_array)
    if h in val_hashes:
        val_duplicates += 1
    else:
        val_hashes.add(h)

# 3. Check Test Split
print("🔍 Checking Test split...")
for i in range(len(ds['test'])):
    img_array = np.array(ds['test'][i]['image'])
    h = get_image_hash(img_array)
    if h in test_hashes:
        test_duplicates += 1
    else:
        test_hashes.add(h)

# 4. Check for Data Leakage (Overlaps)
train_val_overlap = len(train_hashes & val_hashes)
train_test_overlap = len(train_hashes & test_hashes)
val_test_overlap = len(val_hashes & test_hashes)

print("\n" + "="*70)
print("📊 FINAL RESULTS")
print("="*70)
print(f"Train: {len(ds['train']):4d} total | {len(train_hashes):4d} unique | {train_duplicates} internal duplicates")
print(f"Val:   {len(ds['val']):4d} total | {len(val_hashes):4d}   unique | {val_duplicates} internal duplicates")
print(f"Test:  {len(ds['test']):4d} total | {len(test_hashes):4d}  unique | {test_duplicates} internal duplicates")
print("-" * 70)
print(f"⚠️ Train/Val Overlap:  {train_val_overlap} images")
print(f"🚨 Train/Test Overlap: {train_test_overlap} images (DATA LEAKAGE!)")
print(f"⚠️ Val/Test Overlap:   {val_test_overlap} images")
print("="*70)

if train_test_overlap == 0 and train_duplicates == 0 and val_duplicates == 0 and test_duplicates == 0:
    print("✅ SUCCESS: The Hugging Face dataset is 100% CLEAN. No duplicates, no leakage.")
else:
    print("⚠️ WARNING: Duplicates or leakage detected in the uploaded dataset!")

📦 Loading Sartaj dataset DIRECTLY from Hugging Face servers...


data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2360 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/417 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]


SARTAJ DATASET DUPLICATE & LEAKAGE CHECK (VERIFIED FROM HF)
🔍 Checking Train split...
🔍 Checking Val split...
🔍 Checking Test split...

📊 FINAL RESULTS
Train: 2360 total | 2354 unique | 6 internal duplicates
Val:    417 total |  416   unique | 1 internal duplicates
Test:   250 total |  250  unique | 0 internal duplicates
----------------------------------------------------------------------
⚠️ Train/Val Overlap:  1 images
🚨 Train/Test Overlap: 127 images (DATA LEAKAGE!)
⚠️ Val/Test Overlap:   22 images
⚠️ WARNING: Duplicates or leakage detected in the uploaded dataset!


In [12]:
import os
import pandas as pd
import numpy as np
import json
import hashlib
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ==========================================
# 1. CONFIGURATION
# ==========================================
SARTAJ_PATH = Path("/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri") 
HF_REPO_ID = "srijan2715/sartaj-clean-v2" 
SEED = 42
label_to_int = {"glioma": 0, "meningioma": 1, "no": 2, "pituitary": 3}

# ==========================================
# 2. SCAN AND HASH (Fresh Start)
# ==========================================
print("🔍 SCANNING RAW FOLDERS...")
records = []
for split_name in ['Training', 'Testing']:
    split_path = SARTAJ_PATH / split_name
    if not split_path.exists(): continue
    for class_dir in split_path.iterdir():
        if not class_dir.is_dir(): continue
        class_name = class_dir.name.lower().replace("_tumor", "").replace("notumor", "no")
        for img_path in class_dir.glob("*.*"):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                with open(img_path, 'rb') as f:
                    file_hash = hashlib.sha256(f.read()).hexdigest()
                records.append({
                    "split": "train" if split_name == "Training" else "test",
                    "image_path": str(img_path),
                    "label_str": class_name,
                    "hash": file_hash
                })

df = pd.DataFrame(records)
print(f"📊 TOTAL RAW IMAGES FOUND: {len(df)}")

# ==========================================
# 3. THE CRITICAL CLEANING STEP
# ==========================================
print("\n🧹 CLEANING DATA (Removing Leaks & Duplicates)...")
train_hashes = set(df[df['split']=='train']['hash'])

# Remove test images that exist in train
df_no_leak = df[~((df['split'] == 'test') & df['hash'].isin(train_hashes))]
print(f"   - Removed leakage. Count is now: {len(df_no_leak)}")

# Remove internal duplicates
df_clean = df_no_leak.drop_duplicates(subset=['hash'])
print(f"   - Removed internal duplicates. Final clean count: {len(df_clean)}")

if len(df_clean) == len(df):
    print("⚠️ WARNING: Count did not drop! Check your logic.")
else:
    print("✅ SUCCESS: Dataset size reduced successfully.")

# ==========================================
# 4. SPLIT DATA (AFTER CLEANING)
# ==========================================
train_df = df_clean[df_clean['split'] == 'train'].copy()
test_df = df_clean[df_clean['split'] == 'test'].copy()

train_split, val_split = train_test_split(
    train_df, test_size=0.15, stratify=train_df['label_str'], random_state=SEED
)
print(f"\n📐 FINAL SPLITS: Train={len(train_split)}, Val={len(val_split)}, Test={len(test_df)}")

# ==========================================
# 5. PREPROCESS (224x224)
# ==========================================
def process_images(df, split_name):
    images, labels_int = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        if row['label_str'] not in label_to_int: continue
        try:
            img = Image.open(row['image_path']).convert("RGB")
            img_final = img.resize((224, 224), Image.Resampling.BICUBIC)
            images.append(np.array(img_final))
            labels_int.append(label_to_int[row['label_str']])
        except: pass
    return images, labels_int

print("\n🖼️ RESIZING TO 224x224...")
train_imgs, train_lbls = process_images(train_split, "train")
val_imgs, val_lbls = process_images(val_split, "val")
test_imgs, test_lbls = process_images(test_df, "test")

# ==========================================
# 6. UPLOAD
# ==========================================
print("\n🚀 UPLOADING TO HUGGING FACE...")
dataset_dict = DatasetDict({
    "train": Dataset.from_dict({"image": train_imgs, "label": train_lbls}),
    "val": Dataset.from_dict({"image": val_imgs, "label": val_lbls}),
    "test": Dataset.from_dict({"image": test_imgs, "label": test_lbls})
})

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

dataset_dict.push_to_hub(HF_REPO_ID, token=hf_token, private=False, max_shard_size="500MB")
print(f"\n DONE! Link: https://huggingface.co/datasets/{HF_REPO_ID}")

🔍 SCANNING RAW FOLDERS...
📊 TOTAL RAW IMAGES FOUND: 3264

🧹 CLEANING DATA (Removing Leaks & Duplicates)...
   - Removed leakage. Count is now: 3176
   - Removed internal duplicates. Final clean count: 3027
✅ SUCCESS: Dataset size reduced successfully.

📐 FINAL SPLITS: Train=2360, Val=417, Test=250

🖼️ RESIZING TO 224x224...


Processing test: 100%|██████████| 250/250 [00:02<00:00, 104.33it/s]



🚀 UPLOADING TO HUGGING FACE...


Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


 DONE! Link: https://huggingface.co/datasets/srijan2715/sartaj-clean-v2


In [11]:
import numpy as np
import hashlib
from datasets import load_dataset

print("📦 Loading Sartaj dataset DIRECTLY from Hugging Face servers...")
# This forces a download from HF, ignoring any local cache
ds = load_dataset("srijan2715/sartaj-paper-pipeline", download_mode="force_redownload")

def get_image_hash(img_array):
    # Hash the raw bytes of the numpy array to detect exact pixel duplicates
    return hashlib.sha256(img_array.tobytes()).hexdigest()

print("\n" + "="*70)
print("SARTAJ DATASET DUPLICATE & LEAKAGE CHECK (VERIFIED FROM HF)")
print("="*70)

train_hashes = set()
val_hashes = set()
test_hashes = set()

train_duplicates = 0
val_duplicates = 0
test_duplicates = 0

# 1. Check Train Split
print("🔍 Checking Train split...")
for i in range(len(ds['train'])):
    img_array = np.array(ds['train'][i]['image'])
    h = get_image_hash(img_array)
    if h in train_hashes:
        train_duplicates += 1
    else:
        train_hashes.add(h)

# 2. Check Val Split
print("🔍 Checking Val split...")
for i in range(len(ds['val'])):
    img_array = np.array(ds['val'][i]['image'])
    h = get_image_hash(img_array)
    if h in val_hashes:
        val_duplicates += 1
    else:
        val_hashes.add(h)

# 3. Check Test Split
print("🔍 Checking Test split...")
for i in range(len(ds['test'])):
    img_array = np.array(ds['test'][i]['image'])
    h = get_image_hash(img_array)
    if h in test_hashes:
        test_duplicates += 1
    else:
        test_hashes.add(h)

# 4. Check for Data Leakage (Overlaps)
train_val_overlap = len(train_hashes & val_hashes)
train_test_overlap = len(train_hashes & test_hashes)
val_test_overlap = len(val_hashes & test_hashes)

print("\n" + "="*70)
print("📊 FINAL RESULTS")
print("="*70)
print(f"Train: {len(ds['train']):4d} total | {len(train_hashes):4d} unique | {train_duplicates} internal duplicates")
print(f"Val:   {len(ds['val']):4d} total | {len(val_hashes):4d}   unique | {val_duplicates} internal duplicates")
print(f"Test:  {len(ds['test']):4d} total | {len(test_hashes):4d}  unique | {test_duplicates} internal duplicates")
print("-" * 70)
print(f"⚠️ Train/Val Overlap:  {train_val_overlap} images")
print(f"🚨 Train/Test Overlap: {train_test_overlap} images (DATA LEAKAGE!)")
print(f"⚠️ Val/Test Overlap:   {val_test_overlap} images")
print("="*70)

if train_test_overlap == 0 and train_duplicates == 0 and val_duplicates == 0 and test_duplicates == 0:
    print("✅ SUCCESS: The Hugging Face dataset is 100% CLEAN. No duplicates, no leakage.")
else:
    print("⚠️ WARNING: Duplicates or leakage detected in the uploaded dataset!")

📦 Loading Sartaj dataset DIRECTLY from Hugging Face servers...


data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2360 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/417 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]


SARTAJ DATASET DUPLICATE & LEAKAGE CHECK (VERIFIED FROM HF)
🔍 Checking Train split...
🔍 Checking Val split...
🔍 Checking Test split...

📊 FINAL RESULTS
Train: 2360 total | 2354 unique | 6 internal duplicates
Val:    417 total |  416   unique | 1 internal duplicates
Test:   250 total |  250  unique | 0 internal duplicates
----------------------------------------------------------------------
⚠️ Train/Val Overlap:  1 images
🚨 Train/Test Overlap: 127 images (DATA LEAKAGE!)
⚠️ Val/Test Overlap:   22 images
⚠️ WARNING: Duplicates or leakage detected in the uploaded dataset!


In [13]:
import shutil
import numpy as np
import hashlib
from datasets import load_dataset
from huggingface_hub import HfApi

# 1. DELETE LOCAL CACHE (The Magic Trick)
cache_dir = "/root/.cache/huggingface/datasets"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("️ Local cache deleted to force fresh download.")

print("📦 Loading NEW dataset (sartaj-clean-v2) from Hugging Face...")
# 2. LOAD THE NEW REPO
ds = load_dataset("srijan2715/sartaj-clean-v2", download_mode="force_redownload")

def get_image_hash(img_array):
    return hashlib.sha256(img_array.tobytes()).hexdigest()

print("\n" + "="*70)
print("SARTAJ V2 DUPLICATE & LEAKAGE CHECK")
print("="*70)

train_hashes = set()
val_hashes = set()
test_hashes = set()

train_duplicates = val_duplicates = test_duplicates = 0

print("🔍 Checking Train split...")
for i in range(len(ds['train'])):
    h = get_image_hash(np.array(ds['train'][i]['image']))
    if h in train_hashes: train_duplicates += 1
    else: train_hashes.add(h)

print("🔍 Checking Val split...")
for i in range(len(ds['val'])):
    h = get_image_hash(np.array(ds['val'][i]['image']))
    if h in val_hashes: val_duplicates += 1
    else: val_hashes.add(h)

print("🔍 Checking Test split...")
for i in range(len(ds['test'])):
    h = get_image_hash(np.array(ds['test'][i]['image']))
    if h in test_hashes: test_duplicates += 1
    else: test_hashes.add(h)

print("\n" + "="*70)
print("📊 FINAL RESULTS")
print("="*70)
print(f"Train: {len(ds['train']):4d} total | {len(train_hashes):4d} unique | {train_duplicates} internal duplicates")
print(f"Val:   {len(ds['val']):4d} total | {len(val_hashes):4d}   unique | {val_duplicates} internal duplicates")
print(f"Test:  {len(ds['test']):4d} total | {len(test_hashes):4d}  unique | {test_duplicates} internal duplicates")
print("-" * 70)
print(f"️ Train/Val Overlap:  {len(train_hashes & val_hashes)} images")
print(f"🚨 Train/Test Overlap: {len(train_hashes & test_hashes)} images (DATA LEAKAGE!)")
print(f"⚠️ Val/Test Overlap:   {len(val_hashes & test_hashes)} images")
print("="*70)

if len(train_hashes & test_hashes) == 0 and train_duplicates == 0:
    print("✅ SUCCESS: The NEW Hugging Face dataset is 100% CLEAN.")
else:
    print("️ WARNING: Still issues.")

️ Local cache deleted to force fresh download.
📦 Loading NEW dataset (sartaj-clean-v2) from Hugging Face...


README.md:   0%|          | 0.00/536 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2360 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/417 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]


SARTAJ V2 DUPLICATE & LEAKAGE CHECK
🔍 Checking Train split...
🔍 Checking Val split...
🔍 Checking Test split...

📊 FINAL RESULTS
Train: 2360 total | 2354 unique | 6 internal duplicates
Val:    417 total |  416   unique | 1 internal duplicates
Test:   250 total |  250  unique | 0 internal duplicates
----------------------------------------------------------------------
️ Train/Val Overlap:  1 images
🚨 Train/Test Overlap: 127 images (DATA LEAKAGE!)
⚠️ Val/Test Overlap:   22 images
️ WARNING: Still issues.
